# News Article Sentiment Analysis Pipeline

This notebook processes news article titles and uses the FIN-RoBERTa model to calculate sentiment scores. 

**Input**: CSV files with columns (headline, date) - format: `TICKER_news.csv`

**Output**: Daily sentiment per ticker - format: `TICKER_news_sentiment_daily.csv` (columns: `date`, `avg_sentiment`)

**Model**: FIN-RoBERTa (alasteirho/FIN-RoBERTa-Custom) - Custom financial domain-specific RoBERTa

## 1. Imports and Setup

In [1]:
import os
import re
from pathlib import Path
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm
import exchange_calendars

# NYSE exchange calendar — sessions built lazily from data range in map_to_next_session()
nyse_cal = exchange_calendars.get_calendar("XNYS")


def map_to_next_session(bucket_dates_str):
    """Map each bucket date to the first NYSE session >= that date.

    Builds the session array from data min/max + 30-day buffer,
    so it never silently clips if data falls outside a hard-coded range.
    """
    bd = np.array(bucket_dates_str, dtype="datetime64[D]")
    d_min = str(bd.min() - np.timedelta64(30, "D"))
    d_max = str(bd.max() + np.timedelta64(30, "D"))
    sessions = nyse_cal.sessions_in_range(d_min, d_max)
    sess_np = sessions.values.astype("datetime64[D]")
    idx = np.searchsorted(sess_np, bd, side="left")
    idx = np.clip(idx, 0, len(sess_np) - 1)
    return sess_np[idx].astype(str)

c:\Users\alast\miniconda3\envs\fyp-gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Directories definitions
BASE_DIR = Path.cwd().resolve()
if (BASE_DIR / "Raw_Data" / "gdelt_news_data").exists():
    pass
elif (BASE_DIR.parent / "Raw_Data" / "gdelt_news_data").exists():
    BASE_DIR = BASE_DIR.parent

INPUT_DIR = str(BASE_DIR / "Raw_Data" / "gdelt_news_data")
OUTPUT_DIR = BASE_DIR / "Processed_Data" / "news_sentiment_daily"

print(f"BASE_DIR: {BASE_DIR}")
print(f"INPUT_DIR: {INPUT_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

BASE_DIR: C:\Users\alast\OneDrive - University of Greenwich\FYP
INPUT_DIR: C:\Users\alast\OneDrive - University of Greenwich\FYP\Raw_Data\gdelt_news_data
OUTPUT_DIR: C:\Users\alast\OneDrive - University of Greenwich\FYP\Processed_Data\news_sentiment_daily


## 2. Load FIN-RoBERTa Model

In [3]:
# Load FinRoBERTa model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("alasteirho/FIN-RoBERTa-Custom")
model = AutoModelForSequenceClassification.from_pretrained("alasteirho/FIN-RoBERTa-Custom")

# Move model to GPU
device = torch.device("cuda")
model.to(device)
model.eval()
print(f"Model loaded onto {device}")

Model loaded onto cuda


## 3. Preprocessing Function

Cleans news article titles using REG-EX by removing:
- URLs
- Ticker symbols (NASDAQ:XXX format)
- Exchange prefixes
- News source metadata
- Special characters

In [4]:
def preprocess_title(title):
    if pd.isna(title) or not isinstance(title, str):
        return None

    title = re.sub(r'\s+', ' ', title).strip()
    title = re.sub(r'http\S+|www\.\S+', '', title)
    title = re.sub(r'\(\s*(NASDAQ|NYSE)\s*:\s*\w+\s*\)', '', title, flags=re.IGNORECASE)
    title = re.sub(r'\s*[-|]\s*[A-Z]{1,5}\s*$', '', title)
    title = re.sub(r'^\s*(NASDAQ|NYSE)\s*:\s*', '', title, flags=re.IGNORECASE)
    title = re.sub(r'\s*\|\s*[A-Z][a-z]+\s*$', '', title)
    title = re.sub(r'\s*-\s*[A-Z][a-z]+\s+[A-Z][a-z]+\s*$', '', title)
    title = re.sub(r'[^\w\s\.,!?\'\"\-]', ' ', title)
    title = re.sub(r'\s+', ' ', title).strip()

    if len(title) < 15:
        return None

    return title

## 4. Sentiment Scoring

Uses FIN-RoBERTa to calculate sentiment scores from -1 (negative) to +1 (positive)

In [5]:
def get_sentiment_score(texts, batch_size=16):
    if not texts:
        return []

    scores = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.softmax(outputs.logits, dim=1)

        # FinRoBERTa outputs: [negative, neutral, positive]
        # Score = P(positive) - P(negative)
        batch_scores = predictions[:, 2] - predictions[:, 0]
        scores.extend(batch_scores.cpu().numpy().tolist())

    return scores

## 5. File Processing Function

In [6]:
def process_file(input_path, ticker):
    print(f"\nProcessing: {os.path.basename(input_path)} (Ticker: {ticker})")

    df = pd.read_csv(input_path)

    if 'headline' not in df.columns or 'date' not in df.columns:
        print(f"Skipping: Missing 'headline' or 'date' column")
        return

    print("Preprocessing news article titles...")
    df['clean_headline'] = df['headline'].apply(preprocess_title)

    valid_df = df.dropna(subset=['clean_headline']).copy()

    if len(valid_df) == 0:
        print("No valid headlines found after preprocessing")
        return

    print(f"Processing {len(valid_df)} valid news headlines")

    headlines = valid_df['clean_headline'].tolist()
    scores = get_sentiment_score(headlines)
    valid_df['sentiment_score'] = scores

    # Robust UTC timestamp parsing — handles ISO "Z" suffix, tz-naive, etc.
    valid_df['datetime'] = pd.to_datetime(valid_df['date'], utc=True)
    valid_df['sentiment_score'] = valid_df['sentiment_score'].round(4)
    result = valid_df[['datetime', 'sentiment_score']].copy()
    result['ticker'] = ticker
    result = result.sort_values('datetime')
    return result

## 6. Helper Functions

In [7]:
def extract_ticker(filename):
    base = filename.replace('.csv', '')

    if '_news' in base:
        return base.split('_news')[0]
    elif '_' in base:
        parts = base.split('_')
        for part in parts:
            if part.isupper() and 1 <= len(part) <= 5:
                return part

    return None

## 7. Main Processing Pipeline

Steps:
1. Load news CSV files
2. Extract ticker symbols
3. Preprocess titles and calculate sentiment
4. Assign each article to a NYSE trading session using the **16:00 ET market close cutoff**. Articles at or before 16:00 ET belong to that day's session; articles after 16:00 ET belong to the next session. Session mapping uses `exchange_calendars` (XNYS) to correctly skip weekends AND US market holidays.
5. Aggregate to daily `avg_sentiment` per ticker
6. Save sentiment CSVs with columns: `date`, `avg_sentiment`

In [8]:
csv_files = [
    filename 
    for filename in os.listdir(INPUT_DIR) 
    if filename.endswith('.csv')
]

if not csv_files:
    print(f"No CSV files found in {INPUT_DIR}")
else:
    print(f"Found {len(csv_files)} news CSV files to process")

Found 20 news CSV files to process


In [9]:
# Process each news file
results = []
for filename in tqdm(csv_files, desc="Processing news files"):
    input_path = os.path.join(INPUT_DIR, filename)

    ticker = extract_ticker(filename)
    if not ticker:
        print(f"\nSkipping {filename}: Could not extract the ticker symbol")
        continue

    try:
        result = process_file(input_path, ticker)
        if result is not None and not result.empty:
            results.append(result)
    except Exception as e:
        print(f"Error processing {filename}: {e}")

if not results:
    print("\nNo sentiment records generated from news articles.")

Processing news files:   0%|          | 0/20 [00:00<?, ?it/s]


Processing: AAPL_news.csv (Ticker: AAPL)
Preprocessing news article titles...
Processing 5464 valid news headlines


Processing news files:   5%|▌         | 1/20 [00:09<03:06,  9.80s/it]


Processing: AMZN_news.csv (Ticker: AMZN)
Preprocessing news article titles...
Processing 6618 valid news headlines


Processing news files:  10%|█         | 2/20 [00:22<03:27, 11.54s/it]

...

Processing news files:  90%|█████████ | 18/20 [03:37<00:16,  8.32s/it]


Processing: WMT_news.csv (Ticker: WMT)
Preprocessing news article titles...
Processing 2935 valid news headlines


Processing news files:  95%|█████████▌| 19/20 [03:43<00:07,  7.70s/it]


Processing: XOM_news.csv (Ticker: XOM)
Preprocessing news article titles...
Processing 1500 valid news headlines


Processing news files: 100%|██████████| 20/20 [03:46<00:00, 11.34s/it]


In [10]:
# Assign each article to a NYSE trading session via MARKET CLOSE (16:00 ET)
# Market-close bucketing with exchange_calendars for holidays
#   Before 16:00:00 ET  = same calendar date
#   After 16:00:00 ET = next calendar date
#   Then map bucket date : first valid NYSE session >= bucket date
if results:
    combined = pd.concat(results, ignore_index=True)
    combined = combined.dropna(subset=["datetime", "ticker", "sentiment_score"])
    combined = combined.sort_values(["datetime", "ticker"])

    # Ensure UTC-aware, then convert to ET for market-close alignment
    combined["datetime"] = pd.to_datetime(combined["datetime"], utc=True)
    ny = combined["datetime"].dt.tz_convert("America/New_York")

    # Seconds-accurate: 16:00:30 ET is after close
    sec = ny.dt.hour * 3600 + ny.dt.minute * 60 + ny.dt.second
    after_close = sec > 16 * 3600  # strictly after 16:00:00 ET

    # Bucket date: same day if at/before close, next calendar day if after close
    bucket_date = ny.dt.normalize()
    bucket_date = bucket_date.where(~after_close, bucket_date + pd.Timedelta(days=1))
    bucket_dates_str = bucket_date.dt.strftime("%Y-%m-%d").values

    # Map bucket dates to valid NYSE sessions (skips weekends + holidays)
    combined["date"] = map_to_next_session(bucket_dates_str)

    n_same = (~after_close).sum()
    n_next = after_close.sum()
    print(f"Articles assigned via market-close bucketing: {len(combined):,} total")
    print(f"  At/before 16:00 ET (same session): {n_same:,}")
    print(f"  After 16:00 ET (next session):     {n_next:,}")

    # --- Sanity check: 5 sample rows ---
    sample_idx = combined.index[:5]
    _sanity = pd.DataFrame({
        "utc_timestamp": combined.loc[sample_idx, "datetime"].values,
        "et_timestamp": ny.loc[sample_idx].values,
        "trade_date": combined.loc[sample_idx, "date"].values,
    })
    print("\nSanity check : market-close bucketing (16:00 ET cutoff, exchange_calendars XNYS):")
    display(_sanity)

    # --- Daily avg_sentiment only ---
    combined["sentiment_score"] = combined["sentiment_score"].astype(float).clip(-1, 1)

    total_daily_sentiment = (
        combined.groupby(["date", "ticker"])["sentiment_score"]
        .mean()
        .reset_index()
        .rename(columns={"sentiment_score": "avg_sentiment"})
    )
    total_daily_sentiment["avg_sentiment"] = total_daily_sentiment["avg_sentiment"].round(4)
    total_daily_sentiment = total_daily_sentiment.sort_values(["ticker", "date"]).reset_index(drop=True)

    print(f"\nTotal daily sentiment records: {len(total_daily_sentiment)}")
    display(total_daily_sentiment.head())

Articles assigned via market-close bucketing: 90,382 total
  At/before 16:00 ET (same session): 62,102
  After 16:00 ET (next session):     28,280

Sanity check : market-close bucketing (16:00 ET cutoff, exchange_calendars XNYS):


,utc_timestamp,et_timestamp,trade_date
0,2023-08-01 01:15:00,2023-08-01 01:15:00,2023-08-01
1,2023-08-01 01:15:00,2023-08-01 01:15:00,2023-08-01
2,2023-08-01 02:15:00,2023-08-01 02:15:00,2023-08-01
3,2023-08-01 02:15:00,2023-08-01 02:15:00,2023-08-01
4,2023-08-01 03:00:00,2023-08-01 03:00:00,2023-08-01



Total daily sentiment records: 11557


,date,ticker,avg_sentiment
0,2023-08-01,AAPL,0.9956
1,2023-08-02,AAPL,0.6567
2,2023-08-03,AAPL,0.0971
3,2023-08-04,AAPL,-0.0722
4,2023-08-07,AAPL,0.1633


## 8. Save Daily Sentiment CSVs

In [11]:
if results:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    tickers = sorted(total_daily_sentiment["ticker"].unique().tolist())

    for ticker in tickers:
        ticker_sentiment = total_daily_sentiment[
            total_daily_sentiment["ticker"] == ticker
        ][["date", "avg_sentiment"]].copy()

        out_path = OUTPUT_DIR / f"{ticker}_news_sentiment_daily.csv"
        ticker_sentiment.to_csv(out_path, index=False)
        print(f"  {ticker}: {len(ticker_sentiment)} days -> {out_path.name}")

    print(f"\nSaved {len(tickers)} sentiment CSVs to {OUTPUT_DIR}")
    print("Columns: date, avg_sentiment")

  AAPL: 658 days -> AAPL_news_sentiment_daily.csv
  AMZN: 644 days -> AMZN_news_sentiment_daily.csv
  AVGO: 649 days -> AVGO_news_sentiment_daily.csv
  BRK-B: 645 days -> BRK-B_news_sentiment_daily.csv
  GOOGL: 660 days -> GOOGL_news_sentiment_daily.csv
  HD: 481 days -> HD_news_sentiment_daily.csv
  JNJ: 560 days -> JNJ_news_sentiment_daily.csv
  JPM: 629 days -> JPM_news_sentiment_daily.csv
  LLY: 596 days -> LLY_news_sentiment_daily.csv
  MA: 546 days -> MA_news_sentiment_daily.csv
  META: 660 days -> META_news_sentiment_daily.csv
  MSFT: 656 days -> MSFT_news_sentiment_daily.csv
  NVDA: 661 days -> NVDA_news_sentiment_daily.csv
  ORCL: 530 days -> ORCL_news_sentiment_daily.csv
  PG: 446 days -> PG_news_sentiment_daily.csv
  TSLA: 652 days -> TSLA_news_sentiment_daily.csv
  UNH: 543 days -> UNH_news_sentiment_daily.csv
  V: 297 days -> V_news_sentiment_daily.csv
  WMT: 539 days -> WMT_news_sentiment_daily.csv
  XOM: 505 days -> XOM_news_sentiment_daily.csv

Saved 20 sentiment CSVs t